# Sentiment Analysis using Recurrent Neural Networks (LSTM)

**Course: Deep Learning &nbsp;|&nbsp; University Project**

---

## Project Overview

This notebook implements a **complete Sentiment Analysis pipeline from scratch** using PyTorch.
We classify tweets from the **Sentiment140** dataset (1.6 M tweets) as **positive** or **negative**.

### What is built from scratch?
| Component | Method |
|-----------|--------|
| LSTM cell | Raw tensor ops — `torch.matmul`, `torch.sigmoid`, `torch.tanh` |
| Tokenizer | Whitespace split after regex cleaning |
| Vocabulary | Word-frequency counting — no pretrained tokenizer |
| Padding | Manual right-padding to fixed length |

### Dataset
**Sentiment140** — 1.6 million English tweets
Labels: `0` = Negative &nbsp;|&nbsp; `4` = Positive (remapped to 0 / 1)
Source: Kaggle — `kazanova/sentiment140`

## Table of Contents

1. [Install & Import Libraries](#1)
2. [GPU Setup & Random Seeds](#2)
3. [Part A — Custom LSTM Theory](#3)
4. [CustomLSTMCell Implementation](#4)
5. [Comparison with PyTorch nn.LSTMCell](#5)
6. [Part B — Load Dataset](#6)
7. [Data Cleaning](#7)
8. [Tokenization & Vocabulary](#8)
9. [Data Splitting — 60 / 20 / 20](#9)
10. [PyTorch Dataset & DataLoader](#10)
11. [Part C — Model Architecture](#11)
12. [Hyperparameters, Loss & Optimizer](#12)
13. [Training & Validation Loop](#13)
14. [Part D — Visualization](#14)
15. [Part E — Test Evaluation](#15)
16. [Conclusion & References](#16)

In [ ]:
# Install Kaggle API (needed to download the dataset in Colab)
!pip install -q kaggle
print('Installation complete!')

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
import re
import math
import time
from collections import Counter

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine-learning utilities ────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print('All libraries imported successfully!')
print(f'PyTorch : {torch.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device selection ─────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {vram:.1f} GB')
else:
    print('No GPU found — training will be slow.')
    print('Tip: Runtime > Change runtime type > T4 GPU')

---
<a id='3'></a>
## Part A — Custom LSTM Theory

### Why do vanilla RNNs fail on long sequences?

A simple RNN computes:  **h_t = tanh(W · x_t + U · h_{t-1} + b)**

During backpropagation, gradients are multiplied by **W** at every step.
- If |W| < 1 → gradients **vanish** → early tokens are forgotten
- If |W| > 1 → gradients **explode** → training diverges

LSTM fixes this with a **cell state C_t** updated by addition, letting gradients
flow back hundreds of steps unchanged.

---

### Official PyTorch LSTM Equations

At each time step **t**, given input **x_t**, previous hidden state **h_{t-1}**, previous cell state **C_{t-1}**:

```
i_t = σ( W_ii·x_t + b_ii + W_hi·h_{t-1} + b_hi )    ← Input gate
f_t = σ( W_if·x_t + b_if + W_hf·h_{t-1} + b_hf )    ← Forget gate
g_t = tanh( W_ig·x_t + b_ig + W_hg·h_{t-1} + b_hg ) ← Candidate cell
o_t = σ( W_io·x_t + b_io + W_ho·h_{t-1} + b_ho )    ← Output gate

C_t = f_t ⊙ C_{t-1}  +  i_t ⊙ g_t                   ← Cell state update
h_t = o_t ⊙ tanh(C_t)                                ← New hidden state
```

### Gate functions

| Gate | Activation | Output range | Role |
|------|-----------|-------------|------|
| Forget **f_t** | sigmoid | (0, 1) | 0 = erase, 1 = keep old memory |
| Input  **i_t** | sigmoid | (0, 1) | how much new info to write |
| Candidate **g_t** | tanh | (−1, 1) | what new content to write |
| Output **o_t** | sigmoid | (0, 1) | which memory units to expose |

> **Key insight:** C_t is updated with **addition** (`f*C + i*g`), not `tanh(W·C)`.
> Gradients flow straight back through C_t → **no vanishing gradient**.

In [ ]:
# =============================================================================
# CustomLSTMCell
# Implements the official PyTorch LSTM equations from scratch.
# Uses ONLY: torch.matmul  torch.sigmoid  torch.tanh
# Does NOT use nn.LSTMCell or nn.LSTM internally.
# =============================================================================

class CustomLSTMCell(nn.Module):
    '''
    One LSTM time step computed from raw tensor operations.

    Args:
        input_size  : number of features in input vector x_t
        hidden_size : number of features in hidden state h_t

    Call signature:
        h_t, c_t = cell(x_t, h_prev, c_prev)

    Tensor shapes:
        x_t    [B, I]  →  h_t [B, H]
        h_prev [B, H]  →  c_t [B, H]
        c_prev [B, H]
        where B=batch, I=input_size, H=hidden_size
    '''

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size  = input_size
        self.hidden_size = hidden_size

        # ── Input gate weights ────────────────────────────────────────────────
        # W_ii maps x_t     → hidden   shape [H, I]  (transposed in matmul)
        # W_hi maps h_{t-1} → hidden   shape [H, H]
        self.W_ii = nn.Parameter(torch.empty(hidden_size, input_size))
        self.W_hi = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_ii = nn.Parameter(torch.zeros(hidden_size))
        self.b_hi = nn.Parameter(torch.zeros(hidden_size))

        # ── Forget gate weights ───────────────────────────────────────────────
        self.W_if = nn.Parameter(torch.empty(hidden_size, input_size))
        self.W_hf = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_if = nn.Parameter(torch.zeros(hidden_size))
        self.b_hf = nn.Parameter(torch.zeros(hidden_size))

        # ── Candidate cell state weights ──────────────────────────────────────
        self.W_ig = nn.Parameter(torch.empty(hidden_size, input_size))
        self.W_hg = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_ig = nn.Parameter(torch.zeros(hidden_size))
        self.b_hg = nn.Parameter(torch.zeros(hidden_size))

        # ── Output gate weights ───────────────────────────────────────────────
        self.W_io = nn.Parameter(torch.empty(hidden_size, input_size))
        self.W_ho = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_io = nn.Parameter(torch.zeros(hidden_size))
        self.b_ho = nn.Parameter(torch.zeros(hidden_size))

        self._reset_parameters()

    def _reset_parameters(self):
        # Kaiming uniform — same default initialisation as nn.Linear
        for w in [self.W_ii, self.W_hi, self.W_if, self.W_hf,
                  self.W_ig, self.W_hg, self.W_io, self.W_ho]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))

    def forward(self, x_t, h_prev, c_prev):
        # matmul: x_t @ W.T  →  [B,I] @ [I,H]  =  [B,H]

        # ── FORGET GATE ───────────────────────────────────────────────────────
        # How much of the old cell state C_{t-1} should we keep?
        # sigmoid output (0,1):  0 = erase completely,  1 = keep completely
        f_t = torch.sigmoid(
            torch.matmul(x_t, self.W_if.T) + self.b_if +
            torch.matmul(h_prev, self.W_hf.T) + self.b_hf
        )   # [B, H]

        # ── INPUT GATE ────────────────────────────────────────────────────────
        # How much of the new candidate information should we write?
        i_t = torch.sigmoid(
            torch.matmul(x_t, self.W_ii.T) + self.b_ii +
            torch.matmul(h_prev, self.W_hi.T) + self.b_hi
        )   # [B, H]

        # ── CANDIDATE CELL STATE ─────────────────────────────────────────────
        # What new content do we propose writing to memory?
        # tanh output (-1,1): can add or subtract information
        g_t = torch.tanh(
            torch.matmul(x_t, self.W_ig.T) + self.b_ig +
            torch.matmul(h_prev, self.W_hg.T) + self.b_hg
        )   # [B, H]

        # ── OUTPUT GATE ───────────────────────────────────────────────────────
        # Which memory units should be visible in the hidden state?
        o_t = torch.sigmoid(
            torch.matmul(x_t, self.W_io.T) + self.b_io +
            torch.matmul(h_prev, self.W_ho.T) + self.b_ho
        )   # [B, H]

        # ── CELL STATE UPDATE ─────────────────────────────────────────────────
        # Erase old memory  +  write new content
        # Addition (not tanh!) lets gradients flow unchanged → no vanishing
        c_t = f_t * c_prev + i_t * g_t   # [B, H]

        # ── HIDDEN STATE ──────────────────────────────────────────────────────
        # Filtered view of the updated cell state
        h_t = o_t * torch.tanh(c_t)      # [B, H]

        return h_t, c_t

    def init_hidden(self, batch_size, device):
        h0 = torch.zeros(batch_size, self.hidden_size, device=device)
        c0 = torch.zeros(batch_size, self.hidden_size, device=device)
        return h0, c0

    def extra_repr(self):
        return f'input_size={self.input_size}, hidden_size={self.hidden_size}'


# ── Quick shape verification ──────────────────────────────────────────────────
_B, _I, _H = 4, 8, 16
_cell = CustomLSTMCell(input_size=_I, hidden_size=_H)
_x    = torch.randn(_B, _I)
_h0, _c0 = _cell.init_hidden(_B, torch.device('cpu'))
_h1, _c1 = _cell(_x, _h0, _c0)

print('CustomLSTMCell — shape verification')
print(f'  x_t    : {tuple(_x.shape)}   →   input')
print(f'  h_t    : {tuple(_h1.shape)}  →   new hidden state')
print(f'  c_t    : {tuple(_c1.shape)}  →   new cell state')
print(f'  h_t range: [{_h1.min():.3f}, {_h1.max():.3f}]  (tanh-bounded)')
print('PASS')

---
<a id='5'></a>
## Comparison with PyTorch `nn.LSTMCell`

We verify our custom cell is **numerically identical** to PyTorch's built-in by
copying weights from `nn.LSTMCell` into `CustomLSTMCell` and comparing outputs.

### How PyTorch stores weights (fused format)

PyTorch packs all 4 gate weights into **2 large matrices** for GPU efficiency:

```
weight_ih  [4H, I]  →  rows 0:H   = W_ii  (input gate)
                        rows H:2H  = W_if  (forget gate)
                        rows 2H:3H = W_ig  (candidate)
                        rows 3H:4H = W_io  (output gate)

weight_hh  [4H, H]  →  same layout for hidden-side weights
bias_ih    [4H]     →  same layout for input biases
bias_hh    [4H]     →  same layout for hidden biases
```

Gate order is always **i → f → g → o**.

### Steps
1. Create `nn.LSTMCell`, run it on random input
2. Slice `weight_ih`, `weight_hh`, `bias_ih`, `bias_hh` into per-gate tensors
3. Copy those slices into `CustomLSTMCell`
4. Run `CustomLSTMCell` on the same input
5. Compare with `torch.allclose` — differences should be < 1e-5 (float32 rounding only)

In [ ]:
torch.manual_seed(0)
CMP_INPUT  = 8
CMP_HIDDEN = 16
CMP_BATCH  = 4
H = CMP_HIDDEN

# ── Step 1: PyTorch reference cell ───────────────────────────────────────────
pytorch_cell = nn.LSTMCell(input_size=CMP_INPUT, hidden_size=CMP_HIDDEN)

# ── Step 2: Identical random inputs ──────────────────────────────────────────
x_cmp    = torch.randn(CMP_BATCH, CMP_INPUT)
h_cmp    = torch.randn(CMP_BATCH, CMP_HIDDEN)
c_cmp    = torch.randn(CMP_BATCH, CMP_HIDDEN)

# ── Step 3: Run nn.LSTMCell ───────────────────────────────────────────────────
h_pytorch, c_pytorch = pytorch_cell(x_cmp, (h_cmp, c_cmp))

# ── Step 4: Slice fused matrices ─────────────────────────────────────────────
def slice_gate(matrix, gate_idx, H):
    '''Rows [gate_idx*H : (gate_idx+1)*H] belong to one gate.'''
    return matrix[gate_idx * H : (gate_idx + 1) * H].clone().detach()

# input-side weights [H, I]
W_ii = slice_gate(pytorch_cell.weight_ih, 0, H)
W_if = slice_gate(pytorch_cell.weight_ih, 1, H)
W_ig = slice_gate(pytorch_cell.weight_ih, 2, H)
W_io = slice_gate(pytorch_cell.weight_ih, 3, H)

# hidden-side weights [H, H]
W_hi = slice_gate(pytorch_cell.weight_hh, 0, H)
W_hf = slice_gate(pytorch_cell.weight_hh, 1, H)
W_hg = slice_gate(pytorch_cell.weight_hh, 2, H)
W_ho = slice_gate(pytorch_cell.weight_hh, 3, H)

# input-side biases [H]
b_ii = slice_gate(pytorch_cell.bias_ih, 0, H)
b_if = slice_gate(pytorch_cell.bias_ih, 1, H)
b_ig = slice_gate(pytorch_cell.bias_ih, 2, H)
b_io = slice_gate(pytorch_cell.bias_ih, 3, H)

# hidden-side biases [H]
b_hi = slice_gate(pytorch_cell.bias_hh, 0, H)
b_hf = slice_gate(pytorch_cell.bias_hh, 1, H)
b_hg = slice_gate(pytorch_cell.bias_hh, 2, H)
b_ho = slice_gate(pytorch_cell.bias_hh, 3, H)

# ── Step 5: Copy into CustomLSTMCell ─────────────────────────────────────────
custom_cmp = CustomLSTMCell(input_size=CMP_INPUT, hidden_size=CMP_HIDDEN)

with torch.no_grad():   # required when writing directly to nn.Parameter tensors
    custom_cmp.W_ii.copy_(W_ii);  custom_cmp.W_hi.copy_(W_hi)
    custom_cmp.W_if.copy_(W_if);  custom_cmp.W_hf.copy_(W_hf)
    custom_cmp.W_ig.copy_(W_ig);  custom_cmp.W_hg.copy_(W_hg)
    custom_cmp.W_io.copy_(W_io);  custom_cmp.W_ho.copy_(W_ho)
    custom_cmp.b_ii.copy_(b_ii);  custom_cmp.b_hi.copy_(b_hi)
    custom_cmp.b_if.copy_(b_if);  custom_cmp.b_hf.copy_(b_hf)
    custom_cmp.b_ig.copy_(b_ig);  custom_cmp.b_hg.copy_(b_hg)
    custom_cmp.b_io.copy_(b_io);  custom_cmp.b_ho.copy_(b_ho)

# ── Step 6: Run CustomLSTMCell ────────────────────────────────────────────────
h_custom, c_custom = custom_cmp(x_cmp, h_cmp, c_cmp)

# ── Step 7: Numerical comparison ─────────────────────────────────────────────
h_diff = (h_pytorch.detach() - h_custom.detach()).abs()
c_diff = (c_pytorch.detach() - c_custom.detach()).abs()
h_ok   = torch.allclose(h_pytorch.detach(), h_custom.detach(), atol=1e-5)
c_ok   = torch.allclose(c_pytorch.detach(), c_custom.detach(), atol=1e-5)

print('=' * 55)
print('  nn.LSTMCell  vs  CustomLSTMCell')
print('=' * 55)
print(f'  Hidden state  max |diff| : {h_diff.max().item():.2e}')
print(f'  Hidden state mean |diff| : {h_diff.mean().item():.2e}')
print(f'  torch.allclose (h_t)     : {h_ok}')
print(f'  Cell state    max |diff| : {c_diff.max().item():.2e}')
print(f'  Cell state   mean |diff| : {c_diff.mean().item():.2e}')
print(f'  torch.allclose (c_t)     : {c_ok}')
print()
status = 'PASS' if (h_ok and c_ok) else 'FAIL'
print(f'  RESULT: {status} — differences are float32 rounding only (< 1e-5)')

---
<a id='6'></a>
## Part B — Data Preprocessing

### Sentiment140 Dataset

| Property | Value |
|----------|-------|
| Total tweets | 1,600,000 |
| Negative (label 0) | 800,000 |
| Positive (label 4) | 800,000 |
| Language | English |
| Source | Twitter API (2009) |

**Important:** The dataset is **sorted** — all 800k negatives first, then all 800k positives.
Loading only the first N rows gives only negative samples.
Fix: load N/2 rows from each half, then concatenate and shuffle.

### Preprocessing steps
1. Load CSV with pandas
2. Clean tweets with regex
3. Tokenize (split on whitespace)
4. Build vocabulary (word frequency counting)
5. Encode words → integer IDs
6. Pad / truncate to fixed length
7. Split 60 / 20 / 20
8. Create PyTorch `Dataset` + `DataLoader`

In [ ]:
# =============================================================================
# Download Sentiment140 from Kaggle
#
# Setup steps:
#   1. Go to kaggle.com > Account > API > Create New Token
#   2. This downloads kaggle.json to your computer
#   3. Run this cell and upload kaggle.json when prompted
# =============================================================================

import os
from google.colab import files

print('Upload your kaggle.json API key file:')
uploaded = files.upload()   # opens a file-picker dialog

# Place the key where Kaggle CLI expects it
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured.')

# Download and extract
print('Downloading Sentiment140 (238 MB)...')
!kaggle datasets download -d kazanova/sentiment140
!unzip -q sentiment140.zip
print('Done!')
!ls -lh training.1600000.processed.noemoticon.csv

In [ ]:
# =============================================================================
# Load Sentiment140 CSV
# Columns: polarity, id, date, flag, user, text  (no header row in file)
# =============================================================================

CSV_PATH   = 'training.1600000.processed.noemoticon.csv'
N_ROWS     = 100_000    # 50k negative + 50k positive; set None for all 1.6M
TOTAL_ROWS = 1_600_000  # full dataset size

col_names = ['polarity', 'id', 'date', 'flag', 'user', 'text']

# Load equal halves so labels are balanced even at small N_ROWS
half = N_ROWS // 2

neg_df = pd.read_csv(CSV_PATH, encoding='latin-1', header=None,
                     names=col_names, skiprows=0, nrows=half)

pos_df = pd.read_csv(CSV_PATH, encoding='latin-1', header=None,
                     names=col_names, skiprows=TOTAL_ROWS // 2, nrows=half)

df = pd.concat([neg_df, pos_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle

# Keep only polarity and text; remap 4 -> 1
df = df[['polarity', 'text']].copy()
df['label'] = df['polarity'].map({0: 0, 4: 1})
df.drop(columns=['polarity'], inplace=True)

print(f'Rows loaded : {len(df):,}')
print(f'\nLabel counts:')
print(df['label'].value_counts().to_string())
print(f'\nSample rows:')
print(df[['text', 'label']].head(5).to_string(index=False))

---
<a id='7'></a>
## Data Cleaning

Raw tweets contain noise that hurts model learning:

| Noise type | Raw example | After cleaning |
|------------|------------|----------------|
| @mention | `@elonmusk great!` | `great!` |
| URL | `check http://t.co/abc` | `check` |
| Hashtag # | `feeling #happy` | `feeling happy` |
| Punctuation | `can't!!! wow...` | `cant wow` |
| Mixed case | `GREAT day` | `great day` |
| Extra spaces | `too   many  spaces` | `too many spaces` |

### Order matters
1. Remove URLs **first** — they contain `#` and `.` that would confuse later steps
2. Remove `@mentions` — they sometimes precede text we want to keep
3. Strip `#` symbol but **keep the word** — hashtag text carries sentiment
4. Lowercase **after** URL removal — URLs are case-sensitive
5. Remove punctuation **last** — after already handling # and @

In [ ]:
# =============================================================================
# Tweet Cleaning — pre-compiled regex patterns (compiled once, reused millions of times)
# =============================================================================

_URL_PAT     = re.compile(r'https?://\S+|www\.\S+')   # http:// or www.
_MENTION_PAT = re.compile(r'@\w+')                     # @username
_HASHTAG_PAT = re.compile(r'#(\w+)')                   # #word -> keep word
_PUNCT_PAT   = re.compile(r'[^\w\s]')                  # anything not word/space
_SPACE_PAT   = re.compile(r'\s+')                      # multiple whitespace


def clean_tweet(text):
    text = _URL_PAT.sub('', text)           # step 1: remove URLs
    text = _MENTION_PAT.sub('', text)       # step 2: remove @mentions
    text = _HASHTAG_PAT.sub(r'\1', text)    # step 3: #word -> word
    text = text.lower()                     # step 4: lowercase
    text = _PUNCT_PAT.sub(' ', text)        # step 5: remove punctuation
    text = _SPACE_PAT.sub(' ', text)        # step 6: collapse spaces
    return text.strip()                     # step 7: trim edges


# Apply to every tweet
print('Cleaning tweets...')
df['clean_text'] = df['text'].astype(str).apply(clean_tweet)

# Drop tweets that are now empty (were URLs / mentions only)
before = len(df)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df)} empty tweets. Remaining: {len(df):,}')

# Before / after examples
print('\nBefore vs After Cleaning:')
print('-' * 70)
for i in range(3):
    print(f'RAW   : {df["text"].iloc[i]}')
    print(f'CLEAN : {df["clean_text"].iloc[i]}')
    print()

---
<a id='8'></a>
## Tokenization & Vocabulary Building

### Tokenization
Split each cleaned tweet on whitespace → list of word tokens.

```
"love this movie"  →  ["love", "this", "movie"]
```

### Vocabulary (word → integer ID)
```
<PAD>  →  0    # padding token — always index 0
<UNK>  →  1    # unknown words — always index 1
the    →  2    # most frequent word gets index 2
i      →  3
...
love   →  52
```

### Why `min_freq = 2`?
Words appearing only once are usually:
- Typos (`teh`, `hte`)
- Rare proper nouns (`@JohnSmith42`)
They don't help generalisation. Excluding them shrinks the vocabulary
and speeds up training.

### Encoding + Padding
```
["love", "this", "xyz"]  →  [52, 7, 1]    # xyz is UNK (id=1)

Pad to MAX_LEN=30:
[52, 7, 1, 0, 0, 0, ..., 0]               # 27 zeros appended
```

### Critical rule: vocabulary built on TRAINING DATA ONLY
If we build the vocab on all data and then split, val/test word frequencies
leak into the vocabulary — this is **data leakage** that inflates accuracy.

In [ ]:
# =============================================================================
# Vocabulary
# =============================================================================

PAD_TOKEN = '<PAD>'   # always index 0
UNK_TOKEN = '<UNK>'   # always index 1


class Vocabulary:
    '''
    Maps words <-> integer IDs.

    Special tokens are fixed:
        <PAD> -> 0   (used to fill short sequences)
        <UNK> -> 1   (used for words not seen in training)
    '''

    def __init__(self, min_freq=2):
        self.min_freq  = min_freq
        self.word2idx  = {PAD_TOKEN: 0, UNK_TOKEN: 1}
        self.idx2word  = {0: PAD_TOKEN, 1: UNK_TOKEN}

    def build(self, tokenized_series):
        '''
        Count all words across the training split and add
        those appearing >= min_freq times to the vocabulary.
        '''
        counter = Counter()
        for tokens in tokenized_series:
            counter.update(tokens)

        # most_common() is sorted by frequency descending
        # so high-frequency words get low indices
        for word, freq in counter.most_common():
            if freq < self.min_freq:
                break                         # all remaining have freq < min_freq
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word

        print('Vocabulary built:')
        print(f'  Unique words seen    : {len(counter):,}')
        print(f'  Words kept (>={self.min_freq})  : {len(self.word2idx) - 2:,}')
        print(f'  Vocabulary size      : {len(self.word2idx):,}  (incl. PAD + UNK)')

    def encode(self, tokens):
        '''List of word strings  ->  list of integer IDs.'''
        unk = self.word2idx[UNK_TOKEN]
        return [self.word2idx.get(w, unk) for w in tokens]

    def decode(self, ids):
        '''List of integer IDs  ->  list of word strings.'''
        return [self.idx2word.get(i, UNK_TOKEN) for i in ids]

    def __len__(self):
        return len(self.word2idx)


# =============================================================================
# Tokenize + Encode + Pad
# =============================================================================

MAX_LEN  = 30   # sequence length  (median tweet = 12 tokens, nearly all fit in 30)
MIN_FREQ = 2    # vocabulary frequency threshold

# Tokenize: split cleaned text on whitespace
df['tokens'] = df['clean_text'].apply(str.split)

lengths = df['tokens'].apply(len)
print('Token length statistics:')
print(f'  Min    : {lengths.min()}')
print(f'  Max    : {lengths.max()}')
print(f'  Mean   : {lengths.mean():.1f}')
print(f'  Median : {int(lengths.median())}')
pct = (lengths <= MAX_LEN).mean() * 100
print(f'  Tweets fitting in {MAX_LEN} tokens : {pct:.1f}%')
print()

# Vocabulary object (will be fitted on training split only — see next cell)
vocab = Vocabulary(min_freq=MIN_FREQ)


def encode_and_pad(token_series, vocab, max_len):
    '''
    Encode token lists to integer IDs, truncate/pad to max_len.

    Returns:
        LongTensor [N, max_len]
    '''
    pad_id = vocab.word2idx[PAD_TOKEN]   # = 0
    rows = []
    for tokens in token_series:
        ids = vocab.encode(tokens)           # variable-length list of ints
        ids = ids[:max_len]                  # truncate if > max_len
        ids += [pad_id] * (max_len - len(ids))  # pad with 0s if < max_len
        rows.append(ids)
    return torch.tensor(rows, dtype=torch.long)   # [N, max_len]


print('Tokenization done. Vocabulary will be fitted after splitting.')

---
<a id='9'></a>
## Data Splitting — 60 / 20 / 20

### Two-step split with `train_test_split`

```
Full data  [100%]
    │
    ├── test      [20%]       Step 1:  test_size = 0.20
    │
    └── trainval  [80%]
            │
            ├── train  [60%]  Step 2:  test_size = 0.20 / 0.80 = 0.25
            └── val    [20%]
```

### Why `stratify=label`?
Without stratification, random chance could give one split 48% positives and
another 52% — a small but real difference that biases reported metrics.
`stratify=label` guarantees **exactly 50% positive / 50% negative** in every split.

### Why split before building the vocabulary?
Building the vocab after the split, on **training tokens only**, prevents val/test
word information from influencing which words enter the vocabulary.
This is a subtle but important form of **data leakage** to avoid.

In [ ]:
# =============================================================================
# Step 1: 60 / 20 / 20 split
# =============================================================================

# Carve out 20% test set
trainval_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label'],          # preserve 50/50 balance
)

# Split remaining 80% into 60% train + 20% val
# 0.20 / 0.80 = 0.25  of the trainval pool = 20% of the full dataset
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.25,
    random_state=42,
    stratify=trainval_df['label'],
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

total = len(df)
print('Split sizes:')
print(f'  Total  : {total:,}  (100.0%)')
print(f'  Train  : {len(train_df):,}  ({100*len(train_df)/total:.1f}%)')
print(f'  Val    : {len(val_df):,}  ({100*len(val_df)/total:.1f}%)')
print(f'  Test   : {len(test_df):,}  ({100*len(test_df)/total:.1f}%)')

print('\nClass balance (% positive):')
for name, split in [('Train', train_df), ('Val  ', val_df), ('Test ', test_df)]:
    pct = split['label'].mean() * 100
    print(f'  {name} : {pct:.1f}% positive')

# =============================================================================
# Step 2: Build vocabulary — TRAINING DATA ONLY
# =============================================================================
print()
print('Building vocabulary on training data only...')
vocab.build(train_df['tokens'])

# =============================================================================
# Step 3: Encode + pad all three splits using the SAME vocabulary
# =============================================================================
print()
print('Encoding and padding...')
train_ids = encode_and_pad(train_df['tokens'], vocab, MAX_LEN)
val_ids   = encode_and_pad(val_df['tokens'],   vocab, MAX_LEN)
test_ids  = encode_and_pad(test_df['tokens'],  vocab, MAX_LEN)

train_labels = torch.tensor(train_df['label'].values, dtype=torch.float32)
val_labels   = torch.tensor(val_df['label'].values,   dtype=torch.float32)
test_labels  = torch.tensor(test_df['label'].values,  dtype=torch.float32)

print(f'  train_ids : {tuple(train_ids.shape)}')
print(f'  val_ids   : {tuple(val_ids.shape)}')
print(f'  test_ids  : {tuple(test_ids.shape)}')

# Round-trip decode check
sample_tok = train_df['tokens'].iloc[0]
sample_ids = train_ids[0].tolist()
decoded    = vocab.decode([i for i in sample_ids if i != 0])
print(f'\nRound-trip check (train[0]):')
print(f'  Tokens  : {sample_tok}')
print(f'  IDs     : {sample_ids}')
print(f'  Decoded : {decoded}')

---
<a id='10'></a>
## PyTorch Dataset & DataLoader

### `TweetDataset`
A `Dataset` wraps our tensors. PyTorch's `DataLoader` calls `__getitem__`
repeatedly to collect samples into a batch.

### `DataLoader` settings

| Loader | shuffle | Reason |
|--------|---------|--------|
| train_loader | `True` | Prevents model learning sample order |
| val_loader | `False` | Consistent results for comparison |
| test_loader | `False` | Reproducible final evaluation |

### Batch tensor shapes
```
tweet_batch : [batch_size, MAX_LEN]   — LongTensor of token IDs
label_batch : [batch_size]            — FloatTensor of 0.0 / 1.0
```

In [ ]:
# =============================================================================
# TweetDataset
# =============================================================================

class TweetDataset(Dataset):
    '''
    Wraps (token_ids, labels) tensors for use with DataLoader.

    __getitem__ returns one (tweet, label) pair.
    DataLoader calls this repeatedly and stacks them into a batch.
    '''

    def __init__(self, token_ids, labels):
        assert len(token_ids) == len(labels), 'size mismatch'
        self.token_ids = token_ids   # [N, MAX_LEN]  LongTensor
        self.labels    = labels      # [N]            FloatTensor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.token_ids[idx], self.labels[idx]
        # returns: ([MAX_LEN] LongTensor,  scalar FloatTensor)


# =============================================================================
# DataLoaders
# =============================================================================

BATCH_SIZE  = 128
NUM_WORKERS = 2   # parallel loading; set 0 if you see errors

train_dataset = TweetDataset(train_ids, train_labels)
val_dataset   = TweetDataset(val_ids,   val_labels)
test_dataset  = TweetDataset(test_ids,  test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS,
                          pin_memory=(device.type == 'cuda'))

val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

# Verify batch shapes
sample_tw, sample_lb = next(iter(train_loader))
print('DataLoaders ready:')
print(f'  train : {len(train_loader)} batches  ({len(train_dataset):,} samples)')
print(f'  val   : {len(val_loader)} batches  ({len(val_dataset):,} samples)')
print(f'  test  : {len(test_loader)} batches  ({len(test_dataset):,} samples)')
print()
print('One batch shapes:')
print(f'  tweet_batch : {tuple(sample_tw.shape)}   dtype={sample_tw.dtype}')
print(f'  label_batch : {tuple(sample_lb.shape)}   dtype={sample_lb.dtype}')

---
<a id='11'></a>
## Part C — Sentiment Analysis Model

### Architecture

```
token_ids  [batch, seq_len]
    │
    ▼  nn.Embedding  (vocab_size → embed_dim)
embeds  [batch, seq_len, embed_dim]
    │
    ▼  Dropout
    │
    ▼  CustomLSTMCell  ×  seq_len  (unrolled)
    │    at each step t:
    │        x_t = embeds[:, t, :]          [batch, embed_dim]
    │        h_t, c_t = cell(x_t, h, c)    [batch, hidden_size]
    │
    ▼  h_T  (last hidden state)   [batch, hidden_size]
    │
    ▼  Dropout
    │
    ▼  nn.Linear  (hidden_size → 1)
    │
logit  [batch, 1]   ─── BCEWithLogitsLoss applies sigmoid internally
```

### Why the last hidden state?
The LSTM processes the tweet word by word.
At the final step T, **h_T has seen every word** and summarises the whole sentence.
Using h_T for classification is the standard approach for sequence classification.

### Why `BCEWithLogitsLoss`?
It fuses `sigmoid + binary cross-entropy` in one numerically stable operation.
Calling `sigmoid` in the model and `BCELoss` separately can cause NaN for
very large / small logits. `BCEWithLogitsLoss` avoids this.

In [ ]:
# =============================================================================
# SentimentLSTM
# =============================================================================

class SentimentLSTM(nn.Module):
    '''
    Binary sentiment classifier:
        Embedding  -->  CustomLSTMCell (unrolled)  -->  Linear  -->  logit

    Args:
        vocab_size  : size of vocabulary (for nn.Embedding)
        embed_dim   : word embedding dimension
        hidden_size : LSTM hidden state dimension
        n_layers    : number of stacked LSTM layers
        dropout     : dropout rate applied after embedding and between layers
    '''

    def __init__(self, vocab_size, embed_dim, hidden_size,
                 n_layers=1, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_layers    = n_layers

        # Word embeddings: token ID -> dense vector
        # padding_idx=0 keeps the PAD embedding as all-zeros and never updates it
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.drop      = nn.Dropout(dropout)

        # Stack of CustomLSTMCells
        # Layer 0 receives embed_dim input; deeper layers receive hidden_size
        self.lstm_cells = nn.ModuleList([
            CustomLSTMCell(
                input_size  = embed_dim   if layer == 0 else hidden_size,
                hidden_size = hidden_size
            )
            for layer in range(n_layers)
        ])

        # Final linear classifier
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, token_ids):
        '''
        Args:
            token_ids : LongTensor [B, T]   B=batch, T=seq_len

        Returns:
            logits : FloatTensor [B, 1]   (raw score; apply sigmoid for prob)
        '''
        B, T = token_ids.shape
        dev  = token_ids.device

        # Embed token IDs -> [B, T, embed_dim]
        x = self.drop(self.embedding(token_ids))

        # Initialise hidden and cell states to zero for each layer
        h = [torch.zeros(B, self.hidden_size, device=dev) for _ in range(self.n_layers)]
        c = [torch.zeros(B, self.hidden_size, device=dev) for _ in range(self.n_layers)]

        # Unroll LSTM over the sequence one step at a time
        for t in range(T):
            inp = x[:, t, :]                    # [B, embed_dim] — current token
            for l, cell in enumerate(self.lstm_cells):
                h[l], c[l] = cell(inp, h[l], c[l])
                inp = self.drop(h[l])            # feed h as input to the next layer

        # Classify using the last layer's final hidden state
        logits = self.fc(self.drop(h[-1]))       # [B, 1]
        return logits


# ── Build model ───────────────────────────────────────────────────────────────
VOCAB_SIZE  = len(vocab)
EMBED_DIM   = 64
HIDDEN_SIZE = 128
N_LAYERS    = 1
DROPOUT     = 0.3

model = SentimentLSTM(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = EMBED_DIM,
    hidden_size = HIDDEN_SIZE,
    n_layers    = N_LAYERS,
    dropout     = DROPOUT,
).to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'\nTotal parameters     : {total_p:,}')
print(f'Trainable parameters : {trainable_p:,}')

# Forward-pass sanity check
_dummy = torch.randint(0, VOCAB_SIZE, (4, MAX_LEN)).to(device)
_out   = model(_dummy)
print(f'\nForward pass: {tuple(_dummy.shape)} -> {tuple(_out.shape)}  OK')

In [ ]:
# =============================================================================
# Loss, Optimizer, Scheduler
# =============================================================================

EPOCHS   = 10      # number of full passes through training data
LR       = 1e-3    # Adam learning rate
MAX_NORM = 1.0     # gradient-clipping max norm

# BCEWithLogitsLoss: sigmoid + binary cross-entropy, numerically stable
criterion = nn.BCEWithLogitsLoss()

# Adam: per-parameter adaptive learning rates, well-suited for NLP
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Reduce LR by half if val_loss doesn't improve for 2 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5, verbose=True
)

print('Training configuration')
print(f'  Epochs          : {EPOCHS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Learning rate   : {LR}')
print(f'  Optimizer       : Adam')
print(f'  Loss            : BCEWithLogitsLoss')
print(f'  Scheduler       : ReduceLROnPlateau (patience=2)')
print(f'  Grad clipping   : max_norm={MAX_NORM}')
print()
print('Model hyperparameters')
print(f'  Vocab size      : {VOCAB_SIZE:,}')
print(f'  Embed dim       : {EMBED_DIM}')
print(f'  Hidden size     : {HIDDEN_SIZE}')
print(f'  LSTM layers     : {N_LAYERS}')
print(f'  Dropout         : {DROPOUT}')
print(f'  Max seq length  : {MAX_LEN}')
print(f'  Device          : {device}')

In [ ]:
# =============================================================================
# Training and Evaluation Functions
# =============================================================================

def train_epoch(model, loader, criterion, optimizer, device, max_norm):
    '''
    One full pass over the training DataLoader.

    Returns:
        avg_loss : float  — mean loss per sample
        accuracy : float  — fraction of correct predictions
    '''
    model.train()                         # activates Dropout
    total_loss, total_correct, n = 0.0, 0, 0

    for tweets, labels in loader:
        tweets = tweets.to(device)        # [B, T]
        labels = labels.to(device).unsqueeze(1)   # [B, 1]

        optimizer.zero_grad()
        logits = model(tweets)            # [B, 1]
        loss   = criterion(logits, labels)

        loss.backward()

        # Gradient clipping: if the global grad norm > max_norm,
        # scale all gradients down so the norm equals max_norm
        nn.utils.clip_grad_norm_(model.parameters(), max_norm)

        optimizer.step()

        preds = (torch.sigmoid(logits) >= 0.5).float()
        total_correct += (preds == labels).sum().item()
        n             += labels.size(0)
        total_loss    += loss.item() * labels.size(0)

    return total_loss / n, total_correct / n


def evaluate(model, loader, criterion, device):
    '''
    Evaluate model on a DataLoader without updating weights.

    Returns:
        avg_loss   : float
        accuracy   : float
        all_preds  : list of int  (0 or 1 predictions)
        all_labels : list of int  (0 or 1 true labels)
    '''
    model.eval()                          # disables Dropout
    total_loss, total_correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():                 # no gradient computation
        for tweets, labels in loader:
            tweets = tweets.to(device)
            labels = labels.to(device).unsqueeze(1)

            logits = model(tweets)
            loss   = criterion(logits, labels)

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == labels).sum().item()
            n             += labels.size(0)
            total_loss    += loss.item() * labels.size(0)

            all_preds.extend(preds.cpu().squeeze().tolist())
            all_labels.extend(labels.cpu().squeeze().tolist())

    return total_loss / n, total_correct / n, all_preds, all_labels


print('train_epoch() and evaluate() defined.')
print()
print('Training step summary:')
print('  1. model.train()              enable Dropout')
print('  2. forward pass               compute logits [B,1]')
print('  3. BCEWithLogitsLoss          compute scalar loss')
print('  4. loss.backward()            compute gradients')
print('  5. clip_grad_norm_()          prevent gradient explosion')
print('  6. optimizer.step()           update all parameters')

In [ ]:
# =============================================================================
# Full Training Loop
# =============================================================================

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0

print('Starting training...')
print('=' * 72)
print(f'  {"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  '
      f'{"Val Loss":>8}  {"Val Acc":>8}  {"Time":>6}')
print('=' * 72)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Training pass ─────────────────────────────────────────────────────────
    tr_loss, tr_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, MAX_NORM
    )

    # ── Validation pass ───────────────────────────────────────────────────────
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, device)

    elapsed = time.time() - t0

    # Store metrics for plotting
    train_losses.append(tr_loss);  val_losses.append(va_loss)
    train_accs.append(tr_acc);     val_accs.append(va_acc)

    # Adjust learning rate based on validation loss
    scheduler.step(va_loss)

    # Save the best model weights
    tag = ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), 'best_model.pt')
        tag = '  <-- best'

    print(f'  {epoch:>5}  {tr_loss:>10.4f}  {tr_acc*100:>8.2f}%  '
          f'{va_loss:>8.4f}  {va_acc*100:>7.2f}%  {elapsed:>5.1f}s{tag}')

print('=' * 72)
print(f'Training complete.  Best val accuracy: {best_val_acc*100:.2f}%')

# Restore the best weights for evaluation
model.load_state_dict(torch.load('best_model.pt', map_location=device))
print('Best model weights loaded.')

---
<a id='14'></a>
## Part D — Training Visualization

### Four curves to examine

| Plot | What to look for |
|------|-----------------|
| Train loss | Should decrease each epoch |
| Val loss | Should decrease too; rising = overfitting |
| Train accuracy | Should increase toward 100% |
| Val accuracy | True measure of generalisation |

### Diagnosing issues
- **Train loss ↓, Val loss ↑** → overfitting; increase dropout or reduce model size
- **Both losses plateau early** → underfitting; train longer or increase capacity
- **Val accuracy ≈ Train accuracy** → good generalisation

In [ ]:
# =============================================================================
# Training Curves — Loss and Accuracy
# =============================================================================

epochs_x = list(range(1, EPOCHS + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=15, fontweight='bold', y=1.01)

# ── Loss ──────────────────────────────────────────────────────────────────────
ax1.plot(epochs_x, train_losses, 'b-o', label='Train Loss',      linewidth=2, markersize=5)
ax1.plot(epochs_x, val_losses,   'r-o', label='Validation Loss', linewidth=2, markersize=5)
ax1.set_title('Loss per Epoch',     fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Binary Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epochs_x)

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax2.plot(epochs_x, [a*100 for a in train_accs], 'b-o',
         label='Train Accuracy', linewidth=2, markersize=5)
ax2.plot(epochs_x, [a*100 for a in val_accs],   'r-o',
         label='Val Accuracy',   linewidth=2, markersize=5)
ax2.set_title('Accuracy per Epoch', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_x)
ax2.set_ylim([45, 100])

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"Epoch":>5}  {"Train Loss":>10}  {"Train Acc":>9}  '
      f'{"Val Loss":>9}  {"Val Acc":>8}')
print('-' * 48)
for i in range(EPOCHS):
    print(f'{i+1:>5}  {train_losses[i]:>10.4f}  {train_accs[i]*100:>8.2f}%  '
          f'{val_losses[i]:>9.4f}  {val_accs[i]*100:>7.2f}%')

---
<a id='15'></a>
## Part E — Final Evaluation on Test Set

The **test set** was held out during all training and model selection.
It gives the most honest estimate of real-world performance.

### Metrics

| Metric | Formula | Meaning |
|--------|---------|---------|
| Accuracy | (TP+TN) / total | Overall correctness |
| Precision | TP / (TP+FP) | When we predict positive, how often are we right? |
| Recall | TP / (TP+FN) | Of all actual positives, how many did we catch? |
| F1 | 2·P·R / (P+R) | Harmonic mean of precision and recall |

### Confusion matrix

```
                  Predicted
                  Neg    Pos
Actual   Neg  [ TN    FP ]
         Pos  [ FN    TP ]
```

In [ ]:
# =============================================================================
# Test Set Evaluation
# =============================================================================

print('Evaluating on test set...')
test_loss, test_acc, test_preds, test_true = evaluate(
    model, test_loader, criterion, device
)

print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc*100:.2f}%')

# =============================================================================
# Confusion Matrix
# =============================================================================

y_true = [int(l) for l in test_true]
y_pred = [int(p) for p in test_preds]

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

tn, fp, fn, tp = cm.ravel()
print(f'\nConfusion matrix breakdown:')
print(f'  True  Negatives (TN) : {tn:,}')
print(f'  False Positives (FP) : {fp:,}')
print(f'  False Negatives (FN) : {fn:,}')
print(f'  True  Positives (TP) : {tp:,}')

# =============================================================================
# Classification Report
# =============================================================================

print('\nClassification Report:')
print(classification_report(
    y_true, y_pred,
    target_names=['Negative', 'Positive'],
    digits=4
))

---
<a id='16'></a>
## Conclusion

### What was built
| Component | Details |
|-----------|---------|
| CustomLSTMCell | 8 weight matrices, 8 bias vectors; raw matmul + sigmoid + tanh |
| Equivalence proof | `torch.allclose` vs `nn.LSTMCell` with copied weights |
| Tokenizer | Regex cleaning + whitespace split — no external library |
| Vocabulary | Word-frequency counter with `min_freq=2` |
| Data pipeline | Balanced 60/20/20 stratified split; PAD/UNK encoding |
| SentimentLSTM | Embedding → CustomLSTMCell (unrolled) → Linear |
| Training | Adam + ReduceLROnPlateau + gradient clipping |
| Evaluation | Accuracy, confusion matrix, precision/recall/F1 |

### Key insights

**1. LSTM gates solve vanishing gradients**
`C_t = f_t * C_prev + i_t * g_t` — the cell state is updated by addition, not
matrix multiplication, so gradients flow unchanged over hundreds of timesteps.

**2. Data leakage is subtle**
Building the vocabulary on all data before splitting leaks val/test word
frequencies into training. Always split first, then fit preprocessing on train only.

**3. Stratification preserves class balance**
`stratify=label` guarantees the same 50/50 class ratio in every split,
preventing silent metric bias from skewed splits.

**4. CustomLSTMCell == nn.LSTMCell**
After copying weights, `torch.allclose` confirms < 1e-5 difference,
all attributable to float32 rounding order.

---

## Future Improvements

1. **Bidirectional LSTM** — process sequence left-to-right and right-to-left;
   concatenate both final hidden states for richer classification signal.

2. **Multi-layer LSTM** — stack 2–3 layers; each layer refines representations
   from the layer below.

3. **Attention mechanism** — instead of only the last hidden state, learn a
   weighted combination of all hidden states.

4. **Subword tokenisation (BPE)** — handles out-of-vocabulary words better
   than `<UNK>` and reduces vocabulary size.

5. **Pre-trained word embeddings (GloVe)** — initialise the embedding layer
   with GloVe vectors for faster convergence (while still training from scratch).

6. **Data augmentation** — synonym replacement, random deletion to increase
   effective training size without collecting more data.

---

## References

1. Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory*.
   Neural Computation, 9(8), 1735–1780.

2. Go, A., Bhayani, R., & Huang, L. (2009). *Twitter Sentiment Classification
   using Distant Supervision*. Stanford CS224N Technical Report.

3. PyTorch Documentation — `nn.LSTMCell`:
   https://pytorch.org/docs/stable/generated/torch.nn.LSTMCell.html

4. Goodfellow, I., Bengio, Y., & Courville, A. (2016).
   *Deep Learning*. MIT Press. Chapter 10: Sequence Modelling.

5. Srivastava, N. et al. (2014). *Dropout: A Simple Way to Prevent Neural
   Networks from Overfitting*. JMLR, 15(1), 1929–1958.

---

*Notebook implemented from scratch using PyTorch only.*
*No pretrained models. No external tokenisers.*